In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter 

### Text Preprocessing
Columns:

    -'Review Text'
    
    -'Title'

In [2]:
# load the data

# set file path
file_path = "../kaggle_data/Womens Clothing E-Commerce Reviews.csv"

# load with pandas
try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Error: File not found.")


display(df.head())

Dataset loaded successfully!


,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
#rename -> index review column
df.rename(columns={'Unnamed: 0':'rev_idx'}, inplace=True)

#increment
df['rev_idx'] += 1
print(df.head(5))

   rev_idx  Clothing ID  Age                    Title  \
0        1          767   33                      NaN   
1        2         1080   34                      NaN   
2        3         1077   60  Some major design flaws   
3        4         1049   50         My favorite buy!   
4        5          847   47         Flattering shirt   

                                         Review Text  Rating  Recommended IND  \
0  Absolutely wonderful - silky and sexy and comf...       4                1   
1  Love this dress!  it's sooo pretty.  i happene...       5                1   
2  I had such high hopes for this dress and reall...       3                0   
3  I love, love, love this jumpsuit. it's fun, fl...       5                1   
4  This shirt is very flattering to all due to th...       5                1   

   Positive Feedback Count   Division Name Department Name Class Name  
0                        0       Initmates        Intimate  Intimates  
1                        4

In [4]:
#tokenize func
def tokenize(text):
    text = text.strip().lower()
    pattern = r"[a-z]+(?:[-'][a-z]+)?"
    tokens = re.findall(pattern, text)

    return tokens

#remove stopwords func
def remove_stopwords(tokens):
        filtered_words = [word for word in tokens if word not in stopwords]
        return filtered_words

#create custom domain specific stopword list
def domain_stop(words):
    list_words = [word for row in words for word in row] 
    word_counts = Counter(list_words)
    t_20 = word_counts.most_common(20) 
    low_cnt = [word for word, count in word_counts.items() if count == 1]

    return t_20, low_cnt
   

#assign stopwords txt file
with open('../data/stopwords_en.txt', encoding='utf-8') as f: 
    stopwords = f.read()

#handle mising values before tokenize feature columns
df['r_tokens'] = df['Review Text'].fillna('').apply(tokenize)
df['t_tokens'] = df['Title'].fillna('').apply(tokenize)

#remove stopwords
df['r_tokens'] = df['r_tokens'].apply(remove_stopwords)
df['t_tokens'] = df['t_tokens'].apply(remove_stopwords)

#remove domain specific stopwords (high and low frequency)
top_words, low_word_list = domain_stop(df['r_tokens'])
top_title_words, low_title_word_list = domain_stop(df['t_tokens'])

top_20_words = set(word for word, count in top_words)
top_20_title_words = set(word for word, count in top_title_words)

remove_words_review = top_20_words.union(low_word_list)
remove_words_title = top_20_title_words.union(low_title_word_list)

def remove_domain_stopwords(tokens, removals):
    return [word for word in tokens if word not in removals]

df['r_tokens'] = df['r_tokens'].apply(lambda tokens: remove_domain_stopwords(tokens, remove_words_review))
df['t_tokens'] = df['t_tokens'].apply(lambda tokens: remove_domain_stopwords(tokens, remove_words_title))

#remove na's??


In [5]:
#sanity check
print(df.head())


# note: stemming and lemmatization if needed? perhaps lemmatize after and, 
# compare model prefromance downstream?

   rev_idx  Clothing ID  Age                    Title  \
0        1          767   33                      NaN   
1        2         1080   34                      NaN   
2        3         1077   60  Some major design flaws   
3        4         1049   50         My favorite buy!   
4        5          847   47         Flattering shirt   

                                         Review Text  Rating  Recommended IND  \
0  Absolutely wonderful - silky and sexy and comf...       4                1   
1  Love this dress!  it's sooo pretty.  i happene...       5                1   
2  I had such high hopes for this dress and reall...       3                0   
3  I love, love, love this jumpsuit. it's fun, fl...       5                1   
4  This shirt is very flattering to all due to th...       5                1   

   Positive Feedback Count   Division Name Department Name Class Name  \
0                        0       Initmates        Intimate  Intimates   
1                       

## Format and save processed text/data
- cleaned_df.csv (-dataframe (df) saved to csv)
- vmap.txt (unigram, sorted in alphabetical, index from 0)

In [6]:
import os

clean_path = "../data/cleaned_df.csv"
#save cleaned_df.csv
df.to_csv(clean_path, index=False)

parent_dir = os.path.dirname(clean_path)  
vocab_path = os.path.join(parent_dir, "vmap.txt")

# build and save vmap.txt
vocab = sorted(set(
    word 
    for row in df[['r_tokens', 't_tokens']].itertuples(index=False)
    for tokens in row
    for word in tokens
))
voc_map = {word: i for i, word in enumerate(vocab)}

with open(vocab_path, 'w', encoding='utf-8') as f:
    for word, idx in voc_map.items():
        f.write(f"{word}:{idx}\n")

## Summary
Text preprocessing completed on kaggle dataset for downstream nlp modelling, evaluation and deployment. Processed dataset (cleaned_df) and unigram vocabulary (vmap.txt) were saved for use throughout nlp pipeline.